<a href="https://colab.research.google.com/github/peterbabulik/QuantumWalker/blob/main/GroundStateForge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install qiskit cma

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.0/109.0 kB 6.5 MB/s eta 0:00:00


In [2]:

# ==============================================================================
#  The Ground State Forge: An AI-driven simulation of the computational
#  process inside a black hole, as described by the Babulik Inversion.
#
#  Objective: Evolve the fundamental "laws of computation" (a 2-qubit gate)
#  to discover an algorithm that can find the true ground state of a complex
#  quantum system.
# ==============================================================================

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.circuit import ParameterVector, Gate
from qiskit.synthesis.two_qubit import TwoQubitBasisDecomposer
from qiskit.circuit.library import CXGate
from scipy.linalg import expm
import cma
import time

# --- Parameters for the Experiment ---
FORGE_UNIVERSE_SIZE = 6 # The number of qubits for our problem
PAULI_BASIS_2Q = [p1+p2 for p1 in 'IXYZ' for p2 in 'IXYZ' if p1+p2 != 'II']

# ==============================================================================
#  STEP 1: Define the "Infalling Information" (The Problem)
# ==============================================================================

def get_tfim_hamiltonian(n_qubits: int, h: float = 1.0) -> SparsePauliOp:
    """Creates the Hamiltonian for the Transverse-Field Ising Model."""
    zz_terms, x_terms = [], []
    for i in range(n_qubits):
        p_zz = ['I'] * n_qubits
        p_zz[i], p_zz[(i + 1) % n_qubits] = 'Z', 'Z'
        zz_terms.append("".join(p_zz))
        p_x = ['I'] * n_qubits
        p_x[i] = 'X'
        x_terms.append("".join(p_x))
    hamiltonian = SparsePauliOp(zz_terms, coeffs=-np.ones(n_qubits))
    hamiltonian += SparsePauliOp(x_terms, coeffs=-h * np.ones(n_qubits))
    return hamiltonian

def get_true_ground_state(hamiltonian: SparsePauliOp) -> tuple[float, np.ndarray]:
    """Calculates the exact ground state energy and vector via diagonalization."""
    h_matrix = hamiltonian.to_matrix()
    eigvals, eigvecs = np.linalg.eigh(h_matrix)
    return eigvals[0], eigvecs[:, 0]

# ==============================================================================
#  STEP 2: Define the Core Forge and Fitness Functions
# ==============================================================================

def build_gate_from_genes(genes: np.ndarray) -> Gate:
    """Builds a Qiskit Gate from the 15 coefficients provided by the AI."""
    generator_h = SparsePauliOp(PAULI_BASIS_2Q, coeffs=genes)
    u_matrix = expm(-1j * generator_h.to_matrix())
    forged_gate = Gate(name="SolverGate", num_qubits=2, params=[])
    decomposer = TwoQubitBasisDecomposer(CXGate())
    forged_gate.definition = decomposer(u_matrix)
    return forged_gate

def create_vqe_ansatz(gate: Gate, n_qubits: int, depth: int) -> QuantumCircuit:
    """Builds a generic VQE ansatz using the provided gate."""
    ansatz = QuantumCircuit(n_qubits)
    params = ParameterVector('θ', n_qubits * (depth + 1))
    for i in range(n_qubits):
        ansatz.ry(params[i], i)
    for d in range(depth):
        for i in range(n_qubits - 1):
            ansatz.append(gate, [i, i+1])
        for i in range(n_qubits):
            ansatz.ry(params[(d+1)*n_qubits + i], i)
    return ansatz

def ground_state_finder_fitness_function(genes: np.ndarray, hamiltonian: SparsePauliOp) -> float:
    """
    The fitness function. The AI's goal is to minimize this value.
    It returns the lowest energy a gate can find in a fast VQE run.
    """
    try:
        solver_gate = build_gate_from_genes(genes)
        ansatz = create_vqe_ansatz(solver_gate, hamiltonian.num_qubits, depth=1)

        # Inner-loop objective for the VQE
        def vqe_objective(params):
            bound_circuit = ansatz.assign_parameters(params)
            state_vec = Statevector(bound_circuit)
            # Use Qiskit's fast expectation value calculation
            return state_vec.expectation_value(hamiltonian).real

        # Run a quick, inner-loop optimization to test the gate's potential
        x0_inner = np.random.uniform(0, 2 * np.pi, ansatz.num_parameters)
        es_inner = cma.CMAEvolutionStrategy(x0_inner, 0.5, {'bounds': [0, 2*np.pi], 'maxfevals': 100, 'verbose': -9})
        es_inner.optimize(vqe_objective)

        # The cost of the genes is the best energy this gate could find.
        return es_inner.result.fbest

    except Exception:
        return 1e6 # Return a very high energy if anything fails

# ==============================================================================
#  STEP 3: Run the AI Ground State Forge
# ==============================================================================

if __name__ == "__main__":
    print(f"--- The Ground State Forge ---")
    print(f"Objective: Evolve a 2-qubit 'Solver Gate' to find the ground state of a")
    print(f"{FORGE_UNIVERSE_SIZE}-qubit Transverse-Field Ising Model.")

    # --- Setup the problem ---
    problem_hamiltonian = get_tfim_hamiltonian(FORGE_UNIVERSE_SIZE)
    true_gs_energy, true_gs_vector = get_true_ground_state(problem_hamiltonian)
    print(f"\nTarget (True) Ground State Energy: {true_gs_energy:.6f}")

    # --- Setup and run the outer-loop AI ---
    print("\n--- Starting AI Forge to Evolve a Ground State Solver ---")
    x0_outer = np.random.uniform(-np.pi, np.pi, 15)
    sigma0_outer = 0.5
    options_outer = {'bounds': [-np.pi, np.pi], 'maxfevals': 1000, 'verbose': -9}

    # We pass the hamiltonian to the fitness function using a lambda
    es_outer = cma.CMAEvolutionStrategy(x0_outer, sigma0_outer, options_outer)

    start_time = time.time()
    last_print_time = start_time

    while not es_outer.stop():
        solutions = es_outer.ask()
        fitnesses = [ground_state_finder_fitness_function(s, problem_hamiltonian) for s in solutions]
        es_outer.tell(solutions, fitnesses)

        current_time = time.time()
        if current_time - last_print_time > 5:
             print(f"  > Iteration #{es_outer.countiter}, Best Energy Found: {es_outer.result.fbest:.6f}", end='\r')
             last_print_time = current_time

    end_time = time.time()
    print(f"\n  > Forge complete in {end_time - start_time:.2f}s.")

    champion_genes = es_outer.result.xbest

    # ==============================================================================
    #  STEP 4: Analyze the Champion Solver Gate
    # ==============================================================================

    print("\n--- Final Analysis of the Champion 'Solver Gate' ---")

    # Build the final, best gate
    champion_gate = build_gate_from_genes(champion_genes)
    final_ansatz = create_vqe_ansatz(champion_gate, FORGE_UNIVERSE_SIZE, depth=1)

    # Run one final, high-precision VQE to get the best possible result
    print("  > Performing final high-precision VQE with the champion gate...")
    x0_final = np.random.uniform(0, 2 * np.pi, final_ansatz.num_parameters)
    es_final = cma.CMAEvolutionStrategy(x0_final, 0.5, {'bounds': [0, 2*np.pi], 'maxfevals': 500, 'verbose': -9})
    es_final.optimize(lambda p: Statevector(final_ansatz.assign_parameters(p)).expectation_value(problem_hamiltonian).real)

    final_energy_found = es_final.result.fbest
    final_params = es_final.result.xbest
    final_state_vector = Statevector(final_ansatz.assign_parameters(final_params)).data
    final_fidelity = np.abs(np.vdot(true_gs_vector, final_state_vector))**2

    print("\n--- PREDICTION REPORT ---")
    print(f"True Ground State Energy:      {true_gs_energy:.6f}")
    print(f"Energy Found by AI's Solver:   {final_energy_found:.6f}")
    print(f"Fidelity with True Ground State: {final_fidelity:.4%}")

    print("\n--- Genes of the Champion Solver Algorithm ---")
    sorted_genes = sorted(zip(PAULI_BASIS_2Q, champion_genes), key=lambda item: abs(item[1]), reverse=True)
    for pauli, coeff in sorted_genes[:5]:
        print(f"  {pauli}: {coeff:.4f}")

--- The Ground State Forge ---
Objective: Evolve a 2-qubit 'Solver Gate' to find the ground state of a
6-qubit Transverse-Field Ising Model.

Target (True) Ground State Energy: -7.727407

--- Starting AI Forge to Evolve a Ground State Solver ---
  > Iteration #84, Best Energy Found: -6.803119
  > Forge complete in 780.01s.

--- Final Analysis of the Champion 'Solver Gate' ---
  > Performing final high-precision VQE with the champion gate...

--- PREDICTION REPORT ---
True Ground State Energy:      -7.727407
Energy Found by AI's Solver:   -7.355362
Fidelity with True Ground State: 46.5866%

--- Genes of the Champion Solver Algorithm ---
  ZZ: -3.1411
  ZY: -3.0801
  XX: 2.5504
  YZ: -2.4817
  XY: -1.9445
